In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import pandas as pd
import time
from bs4 import BeautifulSoup


# ========================================
# 0. CSV 파일에서 전체 URL 읽기
# ========================================
brands_df = pd.read_csv("무신사_전체_브랜드_tags.csv")

brand_urls = brands_df["URL"].dropna().tolist()

print("총 URL 개수:", len(brand_urls))

# ========================================
# 팝업 제거 함수
# ========================================
def close_popups(driver):
    popup_xpaths = [
        "//button[contains(text(), '확인')]",
        "//button[contains(text(), '오늘 그만보기')]",
        "//button[contains(text(), '닫기')]",
        "//button[contains(text(), 'OK')]"
    ]
    for xp in popup_xpaths:
        try:
            btn = WebDriverWait(driver, 1).until(
                EC.element_to_be_clickable((By.XPATH, xp))
            )
            btn.click()
            time.sleep(0.1)
        except:
            pass



# ========================================
# 브랜드 정보 추출 함수
# ========================================
def extract_brand_info(url):

    start_time = time.time()
    print(f"\n⏳ URL 접속 시작 → {url}")

    # Chrome 실행
    options = Options()
    options.add_argument("--window-size=1400,900")
    driver = webdriver.Chrome(options=options)

    driver.get(url)

    # body가 로딩될 때까지 대기
    WebDriverWait(driver, 7).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    close_popups(driver)

    # 브랜드 정보 버튼 클릭
    info_btn = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//button[@data-button-name='브랜드 정보']"))
    )
    info_btn.click()
    time.sleep(0.5)
    close_popups(driver)

    # 브랜드 정보 전체 박스 로드
    info_wrap = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, "div.Information__Wrap-sc-1j6oprg-0")
        )
    )

    # 브랜드명
    brand_name_el = info_wrap.find_element(
        By.CSS_SELECTOR,
        "div.Information-sc-1j6oprg-3 span.text-title_18px_med.font-pretendard"
    )
    brand_name = BeautifulSoup(brand_name_el.get_attribute("outerHTML"), 'html.parser').get_text().strip()

    # 설명
    desc_el = info_wrap.find_element(
        By.CSS_SELECTOR,
        "span.text-body_13px_reg.text-center.text-gray-600.font-pretendard"
    )
    brand_description = BeautifulSoup(desc_el.get_attribute("outerHTML"), 'html.parser').get_text().strip()

    driver.quit()

    elapsed = time.time() - start_time

    print(f"✔ 브랜드명: {brand_name}")
    print(f"⏱ 처리 시간: {elapsed:.2f}초")

    return brand_name, brand_description



# ========================================
# 🔥 전체 URL 처리 시작
# ========================================
results = []

print("\n========== 전체 브랜드 크롤링 시작 ==========\n")

for idx, url in enumerate(brand_urls, 1):
    print(f"\n[{idx}/{len(brand_urls)}] 진행 중...")

    try:
        name, desc = extract_brand_info(url)
        results.append({
            "brand_name": name,
            "brand_description": desc
        })
    except Exception as e:
        print(f"❌ 실패 → {url}")
        print(f"이유: {e}")
        results.append({
            "brand_name": None,
            "brand_description": None
        })

print("\n========== 전체 크롤링 완료 ==========\n")

# DataFrame 저장 (시간은 포함되지 않음)
df = pd.DataFrame(results)

# 자동 출력은 하지 않음
# print(df)

# 저장 원하면 아래 주석 해제
df.to_csv("무신사_브랜드_설명_결과.csv", index=False, encoding="utf-8-sig")